In [6]:
import os

def setting_up_proxy(proxy=None, proxy_type='http', verbose=True):
    supported_proxy_types = ['http', 'https', 'socks4', 'socks5', 'all']
    assert proxy_type in supported_proxy_types, f"proxy type {repr(proxy_type)} not supported, only support {supported_proxy_types}"
    if proxy is None:
        proxy = os.environ.get(f'{proxy_type}_proxy')
    if proxy is None:
        return
    if verbose:
        print(f'setting up proxy {repr(proxy)} for {repr(proxy_type)}')
    os.environ[f'{proxy_type}_proxy'] = proxy


default_proxy_config = {
    'http': 'http://127.0.0.1:7890',
    'https': 'http://127.0.0.1:7890',
    'all': 'socks5://127.0.0.1:7890',
}


# default_proxy_config = {
#     'http': 'http://10.176.52.116:7890',
#     'https': 'http://10.176.52.116:7890',
#     'all': 'socks5://10.176.52.116:7890',
# }


def setting_up_proxy_from_config(proxy_config=default_proxy_config, verbose=True):
    for proxy_type, proxy_url in proxy_config.items():
        setting_up_proxy(proxy=proxy_url, proxy_type=proxy_type, verbose=verbose)
    print()


# setting_up_proxy_from_config()

In [3]:
import requests
import os
import json
import time

launch_time = time.time()

for i in range(1):
    resp = requests.get('https://www.fangpi.net/download/lrc/1310000', proxies=default_proxy_config)
    # resp = requests.get('https://www.gequbao.net/download/lrc/1310000', proxies=default_proxy_config)
    print(f'{i} {resp} {time.time() - launch_time:.2f}')  # sometimes 429: Too Many Requests
    print(resp)  # sometimes 429: Too Many Requests
    print(resp.content)
    print(resp.headers['content-type'])
    with open('examples/1310000.lrc', 'wb') as f:
        f.write(resp.content)
    # at most 20 reqs in 1 min
    

0 <Response [200]> 1.43
<Response [200]>
b'[00:00]\xe6\x9a\x82\xe6\x97\xa0\xe6\xad\x8c\xe8\xaf\x8d'
text/lrc;charset=UTF-8


In [4]:
import requests
import os
import json
import time

launch_time = time.time()

for i in range(1):
    resp = requests.get('https://www.fangpi.net/api/play_url?id=402856&json=1', proxies=default_proxy_config)
    # resp = requests.get('https://www.gequbao.com/api/play_url?id=402856&json=1', proxies=default_proxy_config)
    print(f'{i} {resp} {time.time() - launch_time:.2f}')  # sometimes 429: Too Many Requests
    print(resp)  # sometimes 429: Too Many Requests
    print(resp.headers['content-type'])
    try:
        json.dump(resp.json(), open('examples/402856-5.json', 'w'), indent=4, ensure_ascii=False)
    except:
        pass
    # at most 10 reqs in 1 min

0 <Response [200]> 1.37
<Response [200]>
application/json


In [11]:
import requests
import os
import json

resp = requests.get('https://www.fangpi.net/music/13510000', proxies=default_proxy_config)
# resp = requests.get('https://www.gequbao.com/music/4420889', proxies=default_proxy_config)
print(resp)  # sometimes 429: Too Many Requests
# print(resp.content)
print(resp.headers['content-type'])

with open('examples/13510000.html', 'wb') as f:
    f.write(resp.content)
# seems no req limit

<Response [200]>
text/html; charset=UTF-8


In [12]:
import re
from RFC.utils.parse import (
    get_html_soup,
    parse,
)


def parse_music_page(resp):
    parse_config = {
        'title': {
            ('attr', 'span', 'class', 'badge badge-pill badge-info', None): {
                ('result', 'text', (('return_as_list', True),), None): {},
            }
        },
        'link': {
            ('attr', 'a', 'id', 'btn-download-mp3', None): {
                # ('result', 'href', None): {},
            }
        },
    }
    title = parse(get_html_soup(resp.content), parse_config['title'])
    link = parse(get_html_soup(resp.content), parse_config['link'], return_str=False)
    print(link)
    try:
        link = link[0]['href']
    except:
        link = ''
    # we delay the format checking to the post-processing stage
    return {
        'title': title,
        'link': link,
    }
    

def parse_music_page_re(resp):
    info = {k: v for k, v in re.findall('window.([0-9a-z_]*) = (.*);', resp.text)}
    info['mp3_lrc'] = [part for part in '\n'.join(re.findall('window.mp3_lrc = `((?:.*)(?:\n.*)*)`;', resp.text, re.MULTILINE)).split('\n') if part]
    return info

info = parse_music_page_re(resp)
import json
print(json.dumps(info, indent=4, ensure_ascii=False))

{
    "mp3_id": "13510000",
    "mp3_url": "''",
    "mp3_title": "'【二胡】画情'",
    "mp3_author": "'永安二胡'",
    "mp3_name": "mp3_title + '-' + mp3_author",
    "mp3_type": "0",
    "mp3_cover": "'/static/img/music_cover.png'",
    "error_msg": "''",
    "mp3_extra_url": "''",
    "mp3_lrc": [
        "[00:00.0]画情 - 乔麦",
        "[00:01.46]词：田辰明",
        "[00:02.92]曲：陈致逸",
        "[00:04.38]留住你一面",
        "[00:07.86]画在我心间",
        "[00:11.64]谁也拿不走",
        "[00:15.37]初见的画面",
        "[00:19.21]哪怕是岁月",
        "[00:22.96]篡改我红颜",
        "[00:26.81]你还是昔日",
        "[00:30.11]多情的少年",
        "[01:08.24]我和你这故事",
        "[01:10.99]只剩皮囊",
        "[01:15.6]恋人早换了模样",
        "[01:23.12]但我紧抓不放",
        "[01:25.94]痛也要逞强",
        "[01:30.58]剩下记忆的猖狂",
        "[01:37.1]不要遗忘",
        "[01:40.95]不要真相",
        "[01:44.66]因为我要",
        "[01:48.3]是你的肩膀",
        "[01:54.259995]留住你一面",
        "[01:57.979996]画在我心间",
        "[02:01.7]谁也拿不走",
        "[02:05.44]初见的画面",
        "[02:09.24]哪怕是岁月",